# Agentic Review with Skills and Helpers

This notebook demonstrates the full agentic capabilities of the v2 API: skills (tools the agent can invoke during review) and helpers (subordinate reviewer agents).

**Skills** give the reviewer access to tools like web search, memory management, and item flagging.
**Helpers** are separate `AgenticReviewer` instances the main reviewer can consult for specialized opinions.

> **All cells in this notebook require a real LLM API key** (e.g., `OPENAI_API_KEY`). Skills and helpers involve multi-step agentic loops that need real model inference.

## Setup

In [ ]:
from lattereview.agentic import AgenticReviewer, ScoringReviewer

# Sample item for review
sample_text = (
    "A novel graph neural network architecture is proposed for drug-drug interaction prediction. "
    "The model was trained on the DrugBank database and validated on 50,000 known interactions. "
    "It achieved an F1 score of 0.94, outperforming existing methods by 8%. "
    "The authors claim the model can predict previously unknown interactions with high confidence."
)

## Enabling Skills

Skills are enabled by passing a list of skill names. Available skills include:

| Skill | Description |
|-------|-------------|
| `"searching-duckduckgo"` | Search the web via DuckDuckGo |
| `"searching-google"` | Search via Google (requires API key) |
| `"searching-pubmed"` | Search PubMed for biomedical literature |
| `"searching-semantic-scholar"` | Search Semantic Scholar |
| `"searching-arxiv"` | Search arXiv preprints |
| `"searching-content"` | Search within provided content |
| `"managing-memory"` | Maintain notes across items |
| `"flagging-items"` | Flag items for human review |
| `"discussing-with-helpers"` | Consult helper agents |

> **Requires API key:** Set `OPENAI_API_KEY` before running.

In [ ]:
# Reviewer with search, memory, and flagging skills
skilled_reviewer = ScoringReviewer(
    name="ResearchAnalyst",
    backstory=(
        "You are a pharmacology AI researcher. When evaluating studies, "
        "you search for related work to verify claims and flag suspicious results."
    ),
    model="openai:gpt-5.4-mini",
    scoring_task="Rate the credibility of this study's claims.",
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules="1=likely false, 2=doubtful, 3=plausible, 4=well-supported, 5=strongly evidenced",
    skills=["searching-duckduckgo", "managing-memory", "flagging-items"],
    agentic_effort="high",
)

result, cost = await skilled_reviewer.review_item(sample_text)
print("Score:", result.get("score"))
print("Reasoning:", result.get("reasoning"))
print(f"Cost: ${cost:.4f}")

## Helper Agents

Helpers are separate `AgenticReviewer` instances that the main reviewer can consult during its agentic loop. This enables multi-agent collaboration.

> **Requires API key:** Both the main reviewer and helpers make LLM calls.

In [ ]:
# Create a domain expert helper
pharmacology_expert = AgenticReviewer(
    name="PharmacologyExpert",
    backstory=(
        "You are a senior pharmacologist with 20 years of experience in drug-drug interactions. "
        "You provide expert opinions on whether reported interaction prediction results are realistic."
    ),
    model="openai:gpt-5.4-mini",
    max_iterations=1,  # Helper uses single-pass for efficiency
)

# Main reviewer with helper access
lead_reviewer = ScoringReviewer(
    name="LeadReviewer",
    backstory="You are a systematic review lead. Consult your pharmacology expert when evaluating drug interaction studies.",
    model="openai:gpt-5.4-mini",
    scoring_task="Rate the methodological quality of this drug interaction study.",
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules="1=poor, 2=fair, 3=good, 4=very good, 5=excellent",
    skills=["discussing-with-helpers"],
    helpers=[pharmacology_expert],
    helper_max_iterations=1,
    agentic_effort="medium",
)

result, cost = await lead_reviewer.review_item(sample_text)
print("Score:", result.get("score"))
print("Reasoning:", result.get("reasoning"))
print(f"Cost: ${cost:.4f}")

## Working Directory Structure

When skills like `managing-memory` or `flagging-items` are used, the reviewer creates files in its working directory. After a run, you'll find:

```
working_dir/
├── memory/           # Notes saved by the managing-memory skill
│   └── notes.json
├── flags/            # Items flagged for human review
│   └── flagged_items.json
└── logs/             # Action logs from the agentic loop
    └── actions.jsonl
```

The memory persists across items, allowing the reviewer to track patterns and accumulate knowledge during a batch review.

In [ ]:
# Review multiple items — memory carries across items
items = [
    "A GNN model predicts drug interactions with F1=0.94 on DrugBank.",
    "Random forest achieves AUC=0.99 on drug interaction prediction using only molecular weight as a feature.",
    "An ensemble approach combining GNN and attention mechanisms achieves F1=0.91 on drug interaction data.",
]

results, cost = await skilled_reviewer.review_items(items)
for i, r in enumerate(results):
    print(f"Item {i+1}: score={r.get('score')}")
print(f"Total cost: ${cost:.4f}")